# Bank Direct Marketing — LightGBM Classification Pipeline

Predicting term deposit subscription (`y`) from the bank direct marketing campaign dataset (41,188 rows, 20 columns).

**Pipeline overview:**
1. Load & clean data
2. Feature engineering (pdays flag, education ordinal, imbalance-aware target)
3. Categorical typing for native LightGBM support
4. Train/validation/test split (stratified)
5. LightGBM training with class-imbalance handling
6. Evaluation (PR-AUC, ROC-AUC, F1, confusion matrix, threshold tuning)
7. Feature importance / SHAP

> ⚠️ **Update `DATA_PATH` below to point to your CSV file before running.**
> This notebook assumes the standard UCI-style bank marketing CSV (semicolon-delimited, columns as listed in the EDA). If your file does NOT have a `duration` column, the drop step below will simply no-op — that's fine.


In [ ]:
# ============================================================
# 0. IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, f1_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================
DATA_PATH = "bank-additional-full.csv"   # <-- UPDATE THIS PATH

# Try semicolon first (standard UCI format), fall back to comma
try:
    df = pd.read_csv(DATA_PATH, sep=';')
    if df.shape[1] == 1:
        raise ValueError("Wrong delimiter")
except Exception:
    df = pd.read_csv(DATA_PATH, sep=',')

print(f"Loaded shape: {df.shape}")
df.head()


## 2. Cleaning

In [ ]:
# ============================================================
# 2a. Drop duplicate rows
# ============================================================
n_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Dropped {n_before - len(df)} duplicate rows ({(n_before - len(df)) / n_before * 100:.2f}%)")
print(f"New shape: {df.shape}")


In [ ]:
# ============================================================
# 2b. Drop leakage feature `duration` if present
# ============================================================
# `duration` (last call length in seconds) is only known AFTER the call
# ends and is near-perfectly predictive as a side effect — it leaks the
# outcome. Never use it for a model meant to predict BEFORE calling.
if 'duration' in df.columns:
    df = df.drop(columns=['duration'])
    print("Dropped 'duration' (leakage feature).")
else:
    print("'duration' not present — nothing to drop.")


In [ ]:
# ============================================================
# 2c. Basic sanity checks
# ============================================================
print("Nulls per column:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution:")
print(df['y'].value_counts(normalize=True))


## 3. Feature engineering

In [ ]:
# ============================================================
# 3a. pdays: separate the "never contacted" placeholder (999) from
#     the genuine numeric value, so the model gets a clean binary
#     signal instead of having to discover the threshold itself.
# ============================================================
df['was_contacted_before'] = (df['pdays'] != 999).astype(int)

# Keep pdays as-is (LightGBM handles the skew fine); the flag above
# gives it an explicit shortcut for the 96%+ "never contacted" case.


In [ ]:
# ============================================================
# 3b. education: encode as ordinal (natural order) rather than
#     nominal, since more schooling is monotonically ordered.
#     Keep the raw column too — LightGBM native categorical splits
#     can still find non-monotonic patterns if they exist.
# ============================================================
education_order = [
    'illiterate', 'basic.4y', 'basic.6y', 'basic.9y',
    'high.school', 'professional.course', 'university.degree', 'unknown'
]
# 'unknown' placed last so it doesn't distort the ordinal scale for
# known levels; the model can still treat it distinctly since it's
# its own integer code.
edu_map = {level: i for i, level in enumerate(education_order)}
df['education_ordinal'] = df['education'].map(edu_map)


In [ ]:
# ============================================================
# 3c. Log-transform heavily skewed count features
#     (LightGBM doesn't strictly need this, but it can slightly help
#     split efficiency on extreme long-tailed counts like `campaign`)
# ============================================================
df['campaign_log'] = np.log1p(df['campaign'])
df['previous_log'] = np.log1p(df['previous'])


In [ ]:
# ============================================================
# 3d. Target to binary
# ============================================================
df['target'] = (df['y'] == 'yes').astype(int)
df = df.drop(columns=['y'])


## 4. Categorical typing for native LightGBM support

In [ ]:
# ============================================================
# 4a. Identify categorical columns and cast to pandas 'category' dtype.
#     'unknown' is kept as a real category (NOT converted to NaN) —
#     it carries genuine signal (e.g. in `default`).
# ============================================================
categorical_cols = [
    'job', 'marital', 'education', 'default', 'housing', 'loan',
    'contact', 'month', 'day_of_week', 'poutcome'
]

for col in categorical_cols:
    df[col] = df[col].astype('category')

print(df.dtypes)


## 5. Train / validation / test split

Stratified on the target to preserve the ~88.7% / 11.3% class ratio across all splits.

In [ ]:
feature_cols = [c for c in df.columns if c != 'target']
X = df[feature_cols]
y = df['target']

# 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f}")
print(f"Val positive rate:   {y_val.mean():.4f}")
print(f"Test positive rate:  {y_test.mean():.4f}")


## 6. LightGBM training

Using `is_unbalance=True` to handle the 7.9:1 class imbalance natively (reweights the loss),
plus early stopping on validation PR-AUC (more informative than accuracy/AUC alone under imbalance).

In [ ]:
train_set = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=categorical_cols,
    free_raw_data=False
)
val_set = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=categorical_cols,
    reference=train_set,
    free_raw_data=False
)

params = {
    'objective': 'binary',
    'metric': ['auc', 'average_precision'],
    'boosting_type': 'gbdt',
    'is_unbalance': True,          # handles class imbalance via internal reweighting
    'learning_rate': 0.03,
    'num_leaves': 31,
    'max_depth': -1,
    'min_data_in_leaf': 30,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1,
    'verbosity': -1,
    'seed': RANDOM_STATE,
}

model = lgb.train(
    params,
    train_set,
    num_boost_round=2000,
    valid_sets=[train_set, val_set],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=100),
    ]
)

print(f"\nBest iteration: {model.best_iteration}")


## 7. Evaluation on held-out test set

In [ ]:
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print(f"Test ROC-AUC: {roc_auc:.4f}")
print(f"Test PR-AUC:  {pr_auc:.4f}   (baseline / random = {y_test.mean():.4f})")


In [ ]:
# ============================================================
# 7a. Threshold tuning — pick the threshold that maximizes F1
#     (swap this objective for a business cost function if you have
#     one, e.g. cost of a call vs. value of a conversion)
# ============================================================
precisions, recalls, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
best_idx = np.argmax(f1_scores[:-1])  # last point has no corresponding threshold
best_threshold = thresholds[best_idx]

print(f"Best F1 threshold: {best_threshold:.4f}")
print(f"  Precision: {precisions[best_idx]:.4f}")
print(f"  Recall:    {recalls[best_idx]:.4f}")
print(f"  F1:        {f1_scores[best_idx]:.4f}")

y_pred_default = (y_pred_proba >= 0.5).astype(int)
y_pred_tuned = (y_pred_proba >= best_threshold).astype(int)

print("\n--- Classification report @ threshold 0.5 ---")
print(classification_report(y_test, y_pred_default, target_names=['no', 'yes']))

print(f"--- Classification report @ tuned threshold {best_threshold:.3f} ---")
print(classification_report(y_test, y_pred_tuned, target_names=['no', 'yes']))


In [ ]:
# ============================================================
# 7b. Confusion matrix (tuned threshold)
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_default, display_labels=['no', 'yes'],
    ax=axes[0], cmap='Blues'
)
axes[0].set_title("Threshold = 0.5")

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_tuned, display_labels=['no', 'yes'],
    ax=axes[1], cmap='Blues'
)
axes[1].set_title(f"Tuned threshold = {best_threshold:.3f}")

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 7c. ROC and Precision-Recall curves
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.4f}')
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

axes[1].plot(recalls, precisions, label=f'PR-AUC = {pr_auc:.4f}')
axes[1].axhline(y=y_test.mean(), linestyle='--', color='gray', label='Random baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Feature importance

In [ ]:
importance_df = pd.DataFrame({
    'feature': model.feature_name(),
    'gain': model.feature_importance(importance_type='gain'),
    'split': model.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=importance_df.head(15), y='feature', x='gain', ax=ax, palette='viridis')
ax.set_title('Top 15 Features by Gain')
plt.tight_layout()
plt.show()

importance_df.head(15)


In [ ]:
# ============================================================
# 8a. (Optional) SHAP values for deeper interpretability
#     Uncomment to run — requires: pip install shap
# ============================================================
# import shap
# explainer = shap.TreeExplainer(model)
# shap_values = explainer.shap_values(X_test)
# shap.summary_plot(shap_values, X_test)


## 9. Cross-validated performance check (robustness)

In [ ]:
# ============================================================
# 5-fold stratified CV on train+val combined, to sanity-check that
# the held-out test performance isn't a lucky split.
# ============================================================
X_cv = pd.concat([X_train, X_val])
y_cv = pd.concat([y_train, y_val])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_pr_aucs, cv_roc_aucs = [], []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cv, y_cv)):
    X_tr, X_va = X_cv.iloc[tr_idx], X_cv.iloc[va_idx]
    y_tr, y_va = y_cv.iloc[tr_idx], y_cv.iloc[va_idx]

    tr_set = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols, free_raw_data=False)
    va_set = lgb.Dataset(X_va, label=y_va, categorical_feature=categorical_cols, reference=tr_set, free_raw_data=False)

    cv_model = lgb.train(
        params, tr_set, num_boost_round=2000,
        valid_sets=[va_set], valid_names=['val'],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=0)]
    )
    preds = cv_model.predict(X_va, num_iteration=cv_model.best_iteration)
    cv_pr_aucs.append(average_precision_score(y_va, preds))
    cv_roc_aucs.append(roc_auc_score(y_va, preds))
    print(f"Fold {fold + 1}: PR-AUC={cv_pr_aucs[-1]:.4f}, ROC-AUC={cv_roc_aucs[-1]:.4f}")

print(f"\nMean PR-AUC:  {np.mean(cv_pr_aucs):.4f} +/- {np.std(cv_pr_aucs):.4f}")
print(f"Mean ROC-AUC: {np.mean(cv_roc_aucs):.4f} +/- {np.std(cv_roc_aucs):.4f}")


## 10. Save the final model

In [ ]:
import os
os.makedirs('models', exist_ok=True)
model.save_model('models/lightgbm_bank_marketing.txt')
print("Model saved to models/lightgbm_bank_marketing.txt")

# To reload later:
# model = lgb.Booster(model_file='models/lightgbm_bank_marketing.txt')


## Notes / next steps

- **Threshold**: the F1-optimal threshold above is a neutral default. If you know the real business cost of a call vs. the value of a conversion, replace the F1 objective in section 7a with an expected-value calculation.
- **Hyperparameter tuning**: the params above are reasonable defaults, not tuned. For a real deployment, run `optuna` or `GridSearchCV`-style search over `num_leaves`, `min_data_in_leaf`, `learning_rate`, and `lambda_l1/l2`.
- **Macro-indicator collinearity**: left untouched here since LightGBM is robust to it, but if you want cleaner feature-importance interpretation, consider dropping `nr.employed` and `emp.var.rate` and keeping only `euribor3m`, then re-running.
- **`is_unbalance` vs `scale_pos_weight`**: these are two different ways to handle imbalance — don't set both. If you want more control than the automatic `is_unbalance` reweighting, replace it with an explicit `scale_pos_weight = (negative_count / positive_count)`.
